# Part 1: Data pipeline (Fernando)

Cleaning (criterion 1) and data parallelism (criterion 3) for our demand forecasting project.
Everything here ends with one table: how many taxi pickups there were in each zone, every hour,
Jan to Apr 2026. Cormac's features and James's benchmarks both start from it.

Data is the NYC yellow taxi trips (`yellow_tripdata_2026-0[1-4].parquet`). It reads from `data/` on a laptop
and from `/home/cs4207/data/` on Rockhill (config.py picks the right one).

## 0. Setup

In [1]:
import config   # has to be imported first, it sets numpy to 1 thread per process
import os, platform, sys, time
import numpy as np, pandas as pd, pyarrow
from src import cleaning as C

start_time = time.perf_counter()

cores = len(os.sched_getaffinity(0)) if hasattr(os, "sched_getaffinity") else os.cpu_count()
print("Python", sys.version.split()[0], "| numpy", np.__version__, "| pandas", pd.__version__,
      "| pyarrow", pyarrow.__version__)
print("Machine:", platform.node(), "|", cores, "cores | on Rockhill:", config.ON_ROCKHILL)
print("Data folder:", config.DATA_DIR)
print("Files:", [f.name for f in config.DATA_FILES])

Python 3.13.18 | numpy 2.5.3 | pandas 3.0.5 | pyarrow 26.0.1
Machine: vm | 2 cores | on Rockhill: False
Data folder: /home/claude/part1-versions/commented/data
Files: ['yellow_tripdata_2026-01.parquet', 'yellow_tripdata_2026-02.parquet', 'yellow_tripdata_2026-03.parquet', 'yellow_tripdata_2026-04.parquet']


## 1. Cleaning

### 1.1 Look at the footer first
The footer is at the end of every Parquet file and tells us the row groups and the min/max pickup
time in each one, without reading any actual rows.

Two things stand out:
- every month is split into 4 row groups, which is where our 16 shards come from (4 months x 4)
- the rows aren't sorted by time. Row group 2 covers the whole month and the first one starts on 31 Dec,
  so filtering by date can't skip anything. Instead we save reading time by only loading the 6 columns we
  actually use, out of 20.

In [2]:
footer = pd.concat([C.footer_summary(f) for f in config.DATA_FILES], ignore_index=True)
footer.to_csv(config.RESULTS_DIR / "footer_row_groups.csv", index=False)
print(f"Reading {len(config.COLUMNS)} of 20 columns: {config.COLUMNS}")
footer

Reading 6 of 20 columns: ['tpep_pickup_datetime', 'tpep_dropoff_datetime', 'PULocationID', 'trip_distance', 'fare_amount', 'payment_type']


,file,row_group,rows,pickup_min,pickup_max
0,yellow_tripdata_2026-01.parquet,0,1048576,2025-12-31 23:57:29,2026-01-13 16:03:09
1,yellow_tripdata_2026-01.parquet,1,1048576,2026-01-13 00:00:00,2026-01-24 19:01:07
2,yellow_tripdata_2026-01.parquet,2,1048576,2026-01-01 00:00:00,2026-02-01 00:45:01
3,yellow_tripdata_2026-01.parquet,3,579161,2026-01-17 17:00:24,2026-01-31 23:59:59
4,yellow_tripdata_2026-02.parquet,0,1048576,2026-01-31 23:31:23,2026-02-12 21:00:42
5,yellow_tripdata_2026-02.parquet,1,1048576,2026-02-12 19:51:16,2026-02-26 09:09:51
6,yellow_tripdata_2026-02.parquet,2,1048576,2026-02-01 00:00:00,2026-03-01 00:51:48
7,yellow_tripdata_2026-02.parquet,3,254138,2026-02-21 17:00:00,2026-02-28 23:59:59
8,yellow_tripdata_2026-03.parquet,0,1048576,2026-02-28 23:38:17,2026-03-11 23:00:29
9,yellow_tripdata_2026-03.parquet,1,1048576,2026-03-11 22:00:01,2026-03-22 07:51:33


### 1.2 Cleaning rules
Checked in this order (`rule_masks()` in `src/cleaning.py`):

| Rule | Dropped if | Why |
|---|---|---|
| outside_month | pickup isn't in that file's month | spill-over into Dec/next month, would get counted twice |
| invalid_zone | zone 264 or 265 | "Unknown" and "Outside NYC" in the zone lookup, nothing to forecast |
| non_positive_fare | fare is 0 or less | refunds and voids, down to -$2,555. Not real new trips |
| bad_distance | 0 miles or over 100 | 0 means it never moved. NYC trips are about 35 mi max, so over 100 is a broken meter (one says 269,097 mi) |
| bad_duration | under 1 minute or over 3 hours | meter started by mistake, or left running. Same limits as the Week 10 example |

**Flex Fare trips** (payment_type 0) are missing passenger count and rate code, but they're real rides and
about a quarter of all trips, so we keep them. We don't use those two columns anyway.

**Hours with no pickups** count as 0, they're not missing. We fill in every zone x hour so the lag
features later on look at the real previous hour.

**Zones** are labels, not numbers, so the model never gets the zone ID as a number.

In [3]:
shards = C.list_shards()
t = time.perf_counter()
demand, shard_stats = C.build_demand_sequential(shards)
serial_s = time.perf_counter() - t
print(f"{len(shards)} shards cleaned and counted one after another in {serial_s:.1f} s")

columns = ["rows_in", *C.RULES, "rows_out", "flex_fare_kept"]
cleaning = shard_stats.groupby("month")[columns].sum()
cleaning.loc["total"] = cleaning.sum()
cleaning["kept_%"] = (100 * cleaning["rows_out"] / cleaning["rows_in"]).round(2)
cleaning.to_csv(config.RESULTS_DIR / "cleaning_before_after.csv")
cleaning

16 shards cleaned and counted one after another in 3.5 s


,rows_in,outside_month,invalid_zone,non_positive_fare,bad_distance,bad_duration,rows_out,flex_fare_kept,kept_%
month,,,,,,,,,
2026-01,3724889,7,5930,41404,121632,55838,3500078,993401,93.96
2026-02,3399866,16,5021,28994,120032,49882,3195921,927292,94.00
2026-03,3952451,19,6249,23612,117653,59910,3745008,858002,94.75
2026-04,3831240,12,6032,17155,90617,60426,3656998,738618,95.45
total,14908446,54,23232,111165,449934,226056,14098005,3517313,94.56


In [4]:
total = cleaning.loc["total"].drop("kept_%").astype(int)
print(f"Before: {total.rows_in:,} trips   After: {total.rows_out:,} trips "
      f"({100 * total.rows_out / total.rows_in:.2f}% kept)")
print(f"Flex Fare trips kept: {total.flex_fare_kept:,} ({100 * total.flex_fare_kept / total.rows_out:.1f}% of clean trips)")
print(f"Zone-hour table: {len(demand):,} rows = {demand.zone.nunique()} zones x {demand.hour.nunique():,} hours")
print(f"Zone-hours with 0 pickups: {100 * (demand.trips == 0).mean():.1f}%")
demand.head()

Before: 14,908,446 trips   After: 14,098,005 trips (94.56% kept)
Flex Fare trips kept: 3,517,313 (24.9% of clean trips)
Zone-hour table: 757,440 rows = 263 zones x 2,880 hours
Zone-hours with 0 pickups: 39.1%


,zone,hour,trips
0,1,2026-01-01 00:00:00,0
1,1,2026-01-01 01:00:00,0
2,1,2026-01-01 02:00:00,0
3,1,2026-01-01 03:00:00,0
4,1,2026-01-01 04:00:00,0


### 1.3 Duplicates and rate codes
Some trips show up twice with the same pickup/drop-off time, zones and distance. In January 35,174 of
the 35,713 are a ride plus a negative-fare copy that refunds it. The fare rule drops the refund and keeps
the ride, and the few left over fail the distance or duration rules, so nothing is double counted.

`RatecodeID` is a category and is blank for Flex Fare trips. We only need pickup time and zone, so we
never read it.

In [5]:
dup_counts, dup_example = C.duplicate_check(config.DATA_FILES[0], config.MONTHS[0])
print(dup_counts)
dup_example[[C.PICKUP, "PULocationID", "DOLocationID", "trip_distance", "fare_amount", "payment_type"]]

{'month': '2026-01', 'rows': 3724889, 'duplicate_rows_before': 35713, 'of_which_pairs_with_a_negative_fare': 35174, 'duplicate_rows_after': 0}


,tpep_pickup_datetime,PULocationID,DOLocationID,trip_distance,fare_amount,payment_type
981,2025-12-31 23:57:29,132,264,0.01,70.0,3
980,2025-12-31 23:57:29,132,264,0.01,-70.0,3
5066,2026-01-01 00:01:19,48,163,0.05,4.4,2
5065,2026-01-01 00:01:19,48,163,0.05,-4.4,2


### 1.4 Train/test split
Split by time, never randomly: Jan to Mar to build the model, April to test it once at the end.
No feature uses anything from later than the hour it's predicting.

## 2. Data parallelism

**Shards:** one per row group, so 4 files x 4 = 16.
- they come straight out of the original files with `read_row_group()`, no need to rewrite anything
- each shard knows its own month, so it can do the spill-over check by itself
- 16 shards and 3 workers: a free worker grabs the next shard (`chunksize=1`). That helps because the last
  row group in each file is smaller.

**Same function on every shard:** `clean_and_count_shard()` reads, cleans and counts.

**Putting it back together:** sort by shard id, then add up the counts. A zone-hour can show up in more than
one shard, so we add instead of overwriting. Counts are whole numbers, so the totals come out exact.

In [6]:
print("Rows per shard:")
print(shard_stats.rows_in.describe()[["min", "mean", "max"]].round(0).to_string())

t = time.perf_counter()
demand_par, stats_par = C.build_demand_parallel(shards, workers=config.MAX_WORKERS)
parallel_s = time.perf_counter() - t

# parallel should give exactly the same table and the same cleaning numbers
assert demand_par.equals(demand) and stats_par.equals(shard_stats)
print(f"\nParallel ({config.MAX_WORKERS} workers): {parallel_s:.1f} s   serial: {serial_s:.1f} s "
      "(the serial run also had to read the files from disk the first time)")
print("Parallel result identical to serial: True")

Rows per shard:
min      254138.0
mean     931778.0
max     1048576.0



Parallel (3 workers): 2.2 s   serial: 3.5 s (the serial run also had to read the files from disk the first time)
Parallel result identical to serial: True


Each worker only ever holds one row group (about 1M rows, 6 columns), never a full month, and sends back a
small table of counts. If a shard was ever too big for memory, `iter_batches()` could break it up further.

## Save the table for the next part

In [7]:
demand.to_parquet(config.RESULTS_DIR / "demand_zone_hour.parquet", index=False)
print(f"Saved {len(demand):,} zone-hours to results/demand_zone_hour.parquet")
print(f"Total run time: {time.perf_counter() - start_time:.1f} s")

Saved 757,440 zone-hours to results/demand_zone_hour.parquet
Total run time: 12.6 s
